# 🏆 Self-Improvement Across Tasks: An Exemplar Memory of Approved Outputs

## Learning Objectives
In this notebook, you will learn:
1. **Intra-task vs inter-task improvement** - A refine loop improves one output; a memory improves the *next* task's first draft
2. **An editor the writer never sees** - House-style rules checked in code, returned as feedback
3. **The refine loop with a memory** - A LangGraph generate → review → revise loop whose generator reads approved examples
4. **Measuring the memory** - First-draft rule failures and revisions needed, with and without the memory, on new tasks
5. **What this is not** - Why this is an RLHF *analogy*: no weights change, and the memory inherits the editor's blind spots

## Where this fits
The generate → critique → revise loop itself is covered in module 09 (`modules/09_evaluator_optimizer/notebooks/01_evaluator_optimizer_langgraph.ipynb`) and module 11 (`modules/11_reflection/notebooks/06_constitutional_ai_critique_loop.ipynb`). Those loops start from zero on every task. This notebook adds the one piece they lack: **keeping the approved results** and feeding them to the generator next time, so the agent's first attempt improves from task to task.

## Prerequisites
- An evaluator-optimizer loop - `modules/09_evaluator_optimizer/notebooks/01_evaluator_optimizer_langgraph.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

| Property | Value |
|---|---|
| Origin | Madaan et al., *Self-Refine* (2023) for the loop; the persistent exemplar memory is an application-level analogy to RLHF, not reinforcement learning |

---
## 📦 Step 1: Environment Setup

Load the API keys from the project-root `.env`. LangSmith tracing is optional: it is used only if your `.env` turns it on.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import os
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Optional: LangSmith traces every LLM call if the .env sets LANGSMITH_TRACING=true.
# Nothing in this notebook depends on it.
tracing = os.getenv("LANGSMITH_TRACING", os.getenv("LANGCHAIN_TRACING_V2", "false")).lower() == "true"
print(f"✅ Environment ready (LangSmith tracing: {'on' if tracing else 'off'})")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: The junior copywriter
# ============================================================================
from aied_helpers import get_openai_llm

# temperature > 0: drafts vary, so the experiment below repeats each run
writer_llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7)

print(f"✅ LLM initialized: {writer_llm.model_name} (temperature={writer_llm.temperature})")

---
## 📖 Part 1: Intra-Task vs Inter-Task Improvement

A **self-improvement loop** evaluates an agent's output and uses the evaluation as feedback. It works on two time scales:

1. **Intra-task (Self-Refine):** draft → critique → revise until the output is approved or a cap is hit. The output improves; the agent does not — the next task starts from scratch.
2. **Inter-task (persistent memory):** approved outputs are stored and shown to the generator on later tasks. The agent's *baseline* improves: first drafts need fewer revisions.

We build a marketing-email writer for a company called Northwind. The **junior copywriter** (the LLM) is never shown the house style guide. Only the **senior editor** knows it — so the writer can learn the style only from feedback and from approved examples.

---
## ✍️ Part 2: The Editor — House Style Checked in Code

Every rule of the style guide is checkable, so the editor is plain Python: no LLM judge whose verdict might drift. It returns one actionable feedback line per broken rule; an email is **approved** when the list is empty.

In [ ]:
# ============================================================================
# EDITOR: Northwind house style, checked in code
# ============================================================================
import re

from pydantic import BaseModel, Field

SIGN_OFF = "— The Northwind Team"


class MarketingEmail(BaseModel):
    subject: str = Field(description="The subject line")
    body: str = Field(description="The plain-text email body")


def editor_review(email: MarketingEmail) -> list:
    """Return one feedback line per broken house-style rule (empty list = approved)."""
    feedback = []
    if len(email.subject) > 45:
        feedback.append(f"The subject is {len(email.subject)} characters; keep it to 45 or fewer.")
    if "!" in email.subject + email.body:
        feedback.append("Remove every exclamation mark; our voice is calm and confident.")
    words = len(email.body.split())
    if not 60 <= words <= 110:
        feedback.append(f"The body is {words} words; it must be 60-110 words.")
    cta_lines = [line for line in email.body.splitlines() if line.strip().startswith("→")]
    if len(cta_lines) != 1:
        feedback.append(f"Use exactly one call-to-action line starting with '→ ' (found {len(cta_lines)}).")
    if not email.body.strip().endswith(SIGN_OFF):
        feedback.append(f"End the body with the sign-off '{SIGN_OFF}'.")
    if not re.search(r"\d", email.body):
        feedback.append("Include one concrete number (a metric, a price, a time saved).")
    return feedback


sample = MarketingEmail(subject="Meet InsightSphere: Revolutionary Analytics for Everyone!",
                        body="Hi there!\nOur new platform is here. Click to learn more.")
for line in editor_review(sample):
    print("✗", line)

---
## 🔁 Part 3: The Refine Loop with a Memory

`GoldStandardMemory` stores approved emails. The graph builder takes a memory as an argument, so the same loop can run **with** it and **without** it (an empty memory) for a fair comparison:

- **`generate`** writes the first draft; its prompt includes the memory's examples (or "none yet").
- **`review`** runs the code editor and writes the feedback to state.
- **`revise`** receives the previous draft **and** the feedback.
- A pure router ends the loop when the feedback is empty or `MAX_REVISIONS` is reached.

In [ ]:
# ============================================================================
# MEMORY + LOOP: generate -> review -> (revise -> review)* -> END
# ============================================================================
from typing import List, Optional, TypedDict

from langgraph.graph import END, START, StateGraph

MAX_REVISIONS = 4


class GoldStandardMemory:
    """Approved emails, shown to the generator as examples of what 'good' looks like."""

    def __init__(self):
        self.examples: List[MarketingEmail] = []

    def add(self, email: MarketingEmail) -> None:
        self.examples.append(email)

    def as_prompt(self) -> str:
        if not self.examples:
            return "(none yet)"
        return "\n\n---\n\n".join(f"Subject: {e.subject}\n{e.body}" for e in self.examples)


class LoopState(TypedDict):
    request: str
    draft: Optional[MarketingEmail]
    feedback: List[str]
    first_draft_failures: int
    revisions: int


writer = writer_llm.with_structured_output(MarketingEmail)


def build_copywriter(memory: GoldStandardMemory, verbose: bool = False):
    def generate(state: LoopState) -> dict:
        draft = writer.invoke(
            "You are a junior marketing copywriter at Northwind.\n"
            f"Previously approved emails (match their style):\n{memory.as_prompt()}\n\n"
            f"Write a marketing email: {state['request']}"
        )
        return {"draft": draft, "revisions": 0}

    def review(state: LoopState) -> dict:
        feedback = editor_review(state["draft"])
        update = {"feedback": feedback}
        if state["revisions"] == 0:
            update["first_draft_failures"] = len(feedback)
        if verbose:
            status = "APPROVED" if not feedback else f"{len(feedback)} rule(s) broken"
            print(f"🧐 review of draft {state['revisions'] + 1}: {status}")
            for line in feedback:
                print(f"     ✗ {line}")
        return update

    def revise(state: LoopState) -> dict:
        feedback = "\n".join(f"- {line}" for line in state["feedback"])
        draft = writer.invoke(
            "You are a junior marketing copywriter at Northwind. Revise your draft to fix every point of the "
            f"editor's feedback.\nRequest: {state['request']}\nDraft subject: {state['draft'].subject}\n"
            f"Draft body:\n{state['draft'].body}\n\nEditor feedback:\n{feedback}"
        )
        return {"draft": draft, "revisions": state["revisions"] + 1}

    def route(state: LoopState) -> str:
        if not state["feedback"] or state["revisions"] >= MAX_REVISIONS:
            return "end"
        return "revise"

    builder = StateGraph(LoopState)
    builder.add_node("generate", generate)
    builder.add_node("review", review)
    builder.add_node("revise", revise)
    builder.add_edge(START, "generate")
    builder.add_edge("generate", "review")
    builder.add_conditional_edges("review", route, {"revise": "revise", "end": END})
    builder.add_edge("revise", "review")
    return builder.compile()


def run(app, request: str) -> dict:
    return app.invoke({"request": request, "feedback": [], "first_draft_failures": 0, "revisions": 0})


print("🧩 Nodes:", [n for n in build_copywriter(GoldStandardMemory()).get_graph().nodes if not n.startswith("__")])

---
## 🌱 Part 4: Seeding the Memory

We run the loop on two briefs with an empty memory and watch the editor's feedback drive the revisions. **Only approved emails are stored** — a draft that hit the revision cap still breaks a rule and must not become an example.

In [ ]:
# ============================================================================
# SEED RUNS: empty memory, full trace; store approved emails only
# ============================================================================
memory = GoldStandardMemory()
seed_app = build_copywriter(memory, verbose=True)

SEED_BRIEFS = [
    "Announce InsightSphere, our new AI-powered data analytics platform.",
    "Announce Pulse, our customer-feedback survey tool.",
]
for brief in SEED_BRIEFS:
    print(f"\n=== {brief} ===")
    result = run(seed_app, brief)
    approved = not result["feedback"]
    print(f"➡️  {result['revisions']} revision(s), approved: {approved}")
    if approved:
        memory.add(result["draft"])

print(f"\n🏆 Memory now holds {len(memory.examples)} approved email(s). The first one:\n")
if memory.examples:
    print(f"Subject: {memory.examples[0].subject}\n{memory.examples[0].body}")

---
## 📊 Part 5: Measuring the Memory on New Tasks

Three **new** briefs. Each is run twice with an empty memory and twice with the seeded memory. We record how many rules the **first draft** broke and how many revisions the loop needed. The memory never tells the writer the rules — the examples only show them.

In [ ]:
# ============================================================================
# EXPERIMENT: first-draft failures and revisions, with vs without memory
# ============================================================================
from concurrent.futures import ThreadPoolExecutor

NEW_BRIEFS = [
    "Promote Visionary, our AI-powered CRM.",
    "Promote Ledger, our invoicing app for freelancers.",
    "Promote Harbor, our cloud backup service.",
]
REPEATS = 2
apps = {"no memory": build_copywriter(GoldStandardMemory()), "memory": build_copywriter(memory)}

jobs = [(label, brief) for brief in NEW_BRIEFS for label in apps for _ in range(REPEATS)]
with ThreadPoolExecutor(max_workers=6) as pool:
    outcomes = list(pool.map(lambda job: run(apps[job[0]], job[1]), jobs))

print(f"{'brief':<52}{'setting':<11}{'first-draft failures':>22}{'revisions':>11}")
summary = {label: {"failures": [], "revisions": [], "approved": 0} for label in apps}
for (label, brief), r in zip(jobs, outcomes):
    print(f"{brief:<52}{label:<11}{r['first_draft_failures']:>22}{r['revisions']:>11}")
    summary[label]["failures"].append(r["first_draft_failures"])
    summary[label]["revisions"].append(r["revisions"])
    summary[label]["approved"] += not r["feedback"]

print()
for label, s in summary.items():
    n = len(s["failures"])
    print(f"{label:<10}: avg first-draft failures {sum(s['failures']) / n:.1f} of 6 rules, "
          f"avg revisions {sum(s['revisions']) / n:.1f}, approved {s['approved']}/{n}")

In our runs, first drafts written without the memory broke 3–6 of the 6 house rules; with two approved examples in the prompt they broke 1–3. The number of **revisions** is much noisier: a revision often fixes one rule and breaks another (watch the seed runs — a draft that fixed its word count gained an exclamation mark), so a better first draft does not always mean fewer rounds. Across our runs the average revisions with memory ranged from fewer (1.3 vs 2.0) to more (2.5 vs 2.0), and a run can even hit the revision cap. The averages printed above are this run's measurement. The loop (Part 3) is what guarantees the rules are eventually met; the memory is what moves the *starting point*.

---
## ⚠️ Part 6: What This Is — and Is Not

- **It is in-context learning.** The model's weights never change; delete the memory and the improvement disappears. Real RLHF trains a reward model on human preferences and updates the policy's weights. The shared intuition — reinforce what was judged good — is why this is only an *analogy*.
- **The memory inherits the editor's blind spots.** Everything stored was approved by this editor. If a rule is wrong or missing, the memory spreads the mistake to every future task. Keep the editor's rules reviewed and the memory auditable (who approved what, when).
- **Memories grow.** Two examples fit in a prompt; two thousand do not. A real system retrieves the few most relevant examples per task (for example by embedding similarity) and prunes stale ones.

---
## 📝 Summary

In this notebook, we learned:

### 1. Intra-task vs inter-task improvement
- A refine loop improves one output; keeping approved outputs improves the next task's starting point.

### 2. An editor the writer never sees
- Checkable house rules belong in code: deterministic verdicts and one actionable feedback line per broken rule.

### 3. The refine loop with a memory
- The builder takes the memory as an argument, so the same loop runs with and without it; only approved drafts are stored.

### 4. Measuring the memory
- On new briefs, first drafts written with approved examples broke far fewer rules; the revision count is noisier, because a revision can fix one rule and break another.

### 5. What this is not
- No weights change (an RLHF analogy, not RLHF), the memory amplifies whatever the editor approves, and a growing memory needs retrieval.

### Next Steps
- **`modules/13_human_approval_basics/notebooks/01_dry_run_harness.ipynb`**: put a human approval step in front of the actions an agent takes